# SafeDecoding — chạy thử trên Google Colab (GPU free)

Chạy lại method **SafeDecoding** (ACL 2024) trên sample nhỏ để **chứng minh method chạy + lấy số DSR**.

## Làm theo đúng thứ tự:
1. Menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save.
2. Chạy **Cell 1 → Cell 2** (cài đặt, ~vài phút).
3. Khi Cell 2 xong, **BẮT BUỘC bấm `Runtime` → `Restart session`** (vì vừa hạ numpy/pyarrow).
4. Sau restart, **KHÔNG chạy lại Cell 1-2**, chạy thẳng từ **Cell 3** trở đi.

_Lần đầu Cell 4 sẽ tải model Vicuna-7B (~13GB) → đợi vài phút._


In [ ]:
# ===== Cell 1: kiểm tra GPU (phải thấy 'Tesla T4'). Nếu lỗi -> chưa bật GPU runtime =====
!nvidia-smi


In [ ]:
# ===== Cell 2: clone repo + cài (chạy 1 lần) =====
# Repo reproduce: code tác giả y bản gốc + requirements đã pin + sample sẵn
!git clone -q https://github.com/PTAxHVA/safedecoding-reproduce.git
%cd /content/safedecoding-reproduce

# Torch 2.1.2 khớp transformers 4.28.1 (bản tác giả). Wheel linux mặc định trên Colab ĐÃ kèm CUDA.
!pip install -q torch==2.1.2
# Phần còn lại đã pin trong requirements.txt (gồm peft bundle đã vá + google-generativeai)
!pip install -q -r requirements.txt

print('='*64)
print('>>> CÀI XONG. BÂY GIỜ bấm  Runtime > Restart session  rồi chạy TIẾP TỪ Cell 3.')
print('>>> (KHÔNG chạy lại Cell 1-2 sau khi restart)')
print('='*64)


## ⚠️ DỪNG LẠI — RESTART RUNTIME
Bấm menu **`Runtime` → `Restart session`** ngay bây giờ (vì vừa hạ numpy/pyarrow; không restart sẽ lỗi import).

Restart xong → chạy thẳng từ **Cell 3** (bỏ qua Cell 1-2). Repo + package đã cài vẫn còn, chỉ kernel khởi động lại.


In [ ]:
# ===== Cell 3: sau RESTART, kiểm tra môi trường =====
%cd /content/safedecoding-reproduce
import torch, transformers, numpy, datasets
print('torch       :', torch.__version__, '| CUDA dùng được:', torch.cuda.is_available())
print('transformers:', transformers.__version__, '(phải 4.28.1)')
print('numpy       :', numpy.__version__, '(phải 1.26.x)')
print('datasets    :', datasets.__version__, '(phải 2.14.6)')
assert torch.cuda.is_available(), 'CHƯA bật GPU! Runtime > Change runtime type > T4 GPU rồi làm lại'


In [ ]:
# ===== Cell 4: CHẠY SafeDecoding (CÓ phòng thủ) trên 20 prompt sample =====
# Lần đầu tải Vicuna-7B (~13GB). Cuối log có dòng  ASR: xx.xx%  ->  DSR = 100 - ASR.
%cd /content/safedecoding-reproduce/exp
!python defense.py --model_name vicuna --attacker custom --defender SafeDecoding --disable_GPT_judge --max_new_tokens 128


In [ ]:
# ===== Cell 5: (đối chứng) TẮT phòng thủ -> DSR sẽ THẤP hơn nhiều =====
!python defense.py --model_name vicuna --attacker custom --defense_off --disable_GPT_judge --max_new_tokens 128


## Đọc kết quả
- Cuối mỗi lần chạy có dòng `ASR: xx.xx%` → **DSR (độ phòng thủ) = 100 − ASR**.
- **Cell 4** (có SafeDecoding) DSR **cao** vs **Cell 5** (tắt) DSR **thấp** → chênh lệch = bằng chứng method có tác dụng.
- File chi tiết ở `/content/safedecoding-reproduce/exp/exp_outputs/...`.

### Nếu lỗi
- **OOM (hết VRAM)**: giảm `--max_new_tokens 64`; hoặc bớt còn 5 prompt — chạy ô này TRƯỚC Cell 4:
  ```python
  import json; p='/content/safedecoding-reproduce/datasets/custom_prompts.json'
  json.dump(json.load(open(p))[:5], open(p,'w'), indent=2)
  ```
- Lỗi khác: copy nguyên log gửi lại để mình sửa.

### (tuỳ chọn) Chấm thêm Harmful Score bằng Gemini
```python
import os; os.environ['GEMINI_API_KEY']='<key>'
!cd /content/safedecoding-reproduce && python gemini_judge.py --input exp/exp_outputs/<thư-mục-vừa-tạo>/<file>.json
```
